# Deep Agents by Langchain

The easiest way to start building agents and applications powered by LLMs—with built-in capabilities for task planning, file systems for context management, subagent-spawning, and long-term memory. You can use deep agents for any task, including complex, multi-step tasks.

We think of deepagents as an “agent harness”. It is the same core tool calling loop as other agent frameworks, but with built-in tools and capabilities.

deepagents is a standalone library built on top of LangChain’s core building blocks for agents. It uses the LangGraph runtime for durable execution, streaming, human-in-the-loop, and other features.

## ReAct Agent: 
Best for straightforward, few-step tasks where the agent needs to call a tool and return an answer immediately (e.g., "What is the weather in Paris?").
## Deep Agent: 
Recommended for "production copilots" and complex, multi-step objectives that require deep research, code execution across multiple files, or long-term persistence. 


In [2]:
!pip install deepagents

  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached websockets-16.0-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached docstring_parser-0.17.0-py3-none-any.whl (36 kB)
   ---------------------------------------- 0.0/760.6 kB ? eta -:--:--
   ---------------------------------------- 760.6/760.6 kB 6.2 MB/s  0:00:00
Using cached websockets-16.0-cp313-cp313-win_amd64.whl (178 kB)
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ------------------ --------------------- 1.6/3.5 MB 7.6 MB/s eta 0:00:01
   --------------------------------- ------ 2.9/3.5 MB 6.9 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 6.7 MB/s  0:00:00
Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl (183 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)

   ----------------------------------

ERROR: Could not install packages due to an OSError: [WinError 183] Cannot create a file when that file already exists: 'C:\\Users\\786\\Desktop\\Sessions\\Agentic AI 2 Month Skill Ustad\\.venv\\Lib\\site-packages\\~angchain_core-1.2.17.dist-info'


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached deepagents-0.5.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached langchain_core-1.2.27-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_anthropic-1.4.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_google_genai-4.2.1-py3-none-any.whl.metadata (2.7 kB)
  Using cached langgraph-1.1.6-py3-none-any.whl.metadata (8.0 kB)
  Using cached google_genai-1.70.0-py3-none-any.whl.metadata (52 kB)
  Using cached google_auth-2.49.1-py3-none-any.whl.metadata (6.2 kB)
  Using cached langgraph_prebuilt-1.0.9-py3-none-any.whl.metadata (5.2 kB)
Using cached deepagents-0.5.1-py3-none-any.whl (123 kB)
Using cached langchain-1.2.15-py3-none-any.whl (112 kB)
Using cached langchain_anthropic-1.4.0-py3-none-any.whl (48 kB)
Using cached langchain_core-1.2.27-py3-none-any.whl (508 kB)
Using cached langchain_google_genai-4.2.1-py3-none-any.whl (66 kB)
Using cached google_genai-1.70.0-py3-none-any.whl (760 k


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
!pip install google-search-results


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import os
from typing import Literal
from deepagents import create_deep_agent
from serpapi.google_search import GoogleSearch
from typing import Literal, List, Dict, Any

In [ ]:


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
) -> List[Dict[str, Any]]:
    """Run a web search using SerpAPI"""
    print(f"Running internet search for query: '{query}' with topic: '{topic}'")
    params = {
        "q": query,
        "api_key": "SERPAPI_KEY",  
        "num": max_results,
    }

    # Topic-based engine selection
    if topic == "news":
        params["tbm"] = "nws"  # Google News
    elif topic == "finance":
        params["tbm"] = "fin"  # Limited support

    try:
        search = GoogleSearch(params)
        results = search.get_dict()
    except Exception as e:
        return [{"error": str(e)}]

    formatted_results = []

    # Extract organic results
    for item in results.get("organic_results", [])[:max_results]:
        result = {
            "title": item.get("title"),
            "link": item.get("link"),
            "snippet": item.get("snippet"),
        }

        if include_raw_content:
            result["raw"] = item

        formatted_results.append(result)
    print(f"Found {len(formatted_results)} results for query: '{query}'")
    print(f"Results: {formatted_results}")
    return formatted_results

In [37]:
# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.
You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

In [22]:
!pip install langchain langchain-google-genai 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    api_key="API_KEY",
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

agent = create_deep_agent(
    model=llm,
    tools=[internet_search],
    system_prompt=research_instructions,
)

In [47]:
result = agent.invoke({"messages": [{"role": "user", "content": "Who is Zeehan Ali Engineer?"}]})



Running internet search for query: 'Zeehan Ali Engineer' with topic: 'general'
Found 5 results for query: 'Zeehan Ali Engineer'
Results: [{'title': 'Zeeshan Ali - Software Engineer | Ex-Tesla', 'link': 'https://www.linkedin.com/in/alizeeshan7', 'snippet': "As a Staff Software Engineer, I've led cross-functional teams to design and implement scalable, multi-tenant platforms, driving key product and platform ..."}, {'title': 'Zeeshan Ali | Project Lead & AI Software Engineer Portfolio', 'link': 'https://zeeshan.p2pclouds.net/', 'snippet': 'Zeeshan Ali is a Practical Technologist & Project Lead at P2P Clouds and Instructor at NeXskill, Ideoversity. Specializing in AI, Blockchain, ...'}, {'title': 'Zeeshan Ali (Engineer) (@mr_zeee_ali)', 'link': 'https://www.instagram.com/mr_zeee_ali/', 'snippet': 'Zeeshan Ali (Engineer). 238 followers. 132 following. Human being. #raqsebismil #instagramreels #reels #dubai #lifeindubai #abudhabi ...'}, {'title': 'Zeeshan Ali — Principal Frontend Engineer',

In [45]:
# Print the agent's response
print(result["messages"][-1].content[0]["text"])

There are several individuals named Zeeshan Ali who are engineers, and it's possible "Zeehan" is a slight misspelling. Here's a summary of some prominent individuals with that name:

*   **Zeeshan Ali (Ex-Tesla):** A Staff Software Engineer who has led cross-functional teams in designing and implementing scalable, multi-tenant platforms.
*   **Zeeshan Ali (P2P Clouds & NeXskill):** A Practical Technologist and Project Lead at P2P Clouds, and an Instructor at NeXskill, Ideoversity, specializing in AI and Blockchain.
*   **Zeeshan Ali (Principal Frontend Engineer):** A Principal Frontend Engineer with over 9 years of experience in designing, developing, and scaling enterprise-grade, data-intensive web applications.
*   **Zeeshan Ali (AWS):** A Systems Development Engineer at AWS, skilled in problem-solving, operating systems, and distributed systems.
